In [0]:
from pyspark.sql import Window, functions as F

silver = "workspace.silver.adidas_us_sales"
gold = "workspace.gold"
checkpoint = "/Volumes/workspace/gold/fact_ventas/checkpoint"

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.gold.fact_ventas")


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold}.fact_ventas (
        id_venta BIGINT, id_tienda STRING, id_ciudad STRING, id_producto STRING,
        precio_unitario INT, unidades_vendidas INT, total_ventas INT, ganancia_operativa INT,
        id_canal_venta STRING, id_calendario BIGINT
    ) USING DELTA
""")

In [0]:
def ultimos_cambios(lote, clave):
    eventos = lote.filter(F.col("_change_type") != "update_preimage")
    orden = Window.partitionBy(clave).orderBy(F.col("_commit_version").desc())
    return (
        eventos.withColumn("_orden", F.row_number().over(orden))
        .filter("_orden = 1")
        .drop("_orden", "_commit_version", "_commit_timestamp")
    )


In [0]:
def aplicar_cambios(datos, destino, clave, columnas):
    sesion = datos.sparkSession
    datos.createOrReplaceTempView("cambios_pipeline")
    asignaciones = ", ".join(f"t.{c} = s.{c}" for c in columnas if c != clave)
    diferencias = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in columnas if c != clave)
    nombres = ", ".join(columnas)
    valores = ", ".join(f"s.{c}" for c in columnas)
    sesion.sql(f"""
        MERGE INTO {destino} t USING cambios_pipeline s ON t.{clave} = s.{clave}
        WHEN MATCHED AND s._borrar THEN DELETE
        WHEN MATCHED AND NOT s._borrar AND ({diferencias}) THEN UPDATE SET {asignaciones}
        WHEN NOT MATCHED AND NOT s._borrar THEN INSERT ({nombres}) VALUES ({valores})
    """)

In [0]:
def actualizar_gold(lote, _):
    if lote.isEmpty():
        return

    cambios = (
        ultimos_cambios(lote, "id_venta")
        .withColumn("id_tienda", F.sha2("tienda", 256))
        .withColumn("id_ciudad", F.sha2(F.to_json(F.struct("ciudad", "estado", "region")), 256))
        .withColumn("id_producto", F.sha2("producto", 256))
        .withColumn("id_canal_venta", F.sha2("canal_venta", 256))
        .withColumn("id_calendario", F.date_format("fecha_factura", "yyyyMMdd").cast("long"))
    )
    columnas_fact = [
        "id_venta", "id_tienda", "id_ciudad", "id_producto", "precio_unitario",
        "unidades_vendidas", "total_ventas", "ganancia_operativa", "id_canal_venta", "id_calendario",
    ]
    hechos = cambios.select(*columnas_fact, F.expr("_change_type = 'delete' AS _borrar"))
    aplicar_cambios(hechos, f"{gold}.fact_ventas", "id_venta", columnas_fact)


flujo = (
    spark.readStream.option("readChangeFeed", "true").table(silver)
    .writeStream.option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .foreachBatch(actualizar_gold)
    .start()
)
flujo.awaitTermination()
